## Leectura De Datos Y Librerias


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from os import path
from scipy.stats import shapiro
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.stats import kruskal

df_results = pd.read_csv("results.zip", low_memory=False)
df_schema = pd.read_csv("schema.csv", low_memory=False)

## "schema": 
**Es el diccionario de datos de la encuesta. `results.zip` tiene las respuestas de cada persona, pero con nombres de columna abreviados. `schema.csv` te dice qué pregunta exacta corresponde a cada una de esas columnas.**

## "results":
**Contiene las respuestas individuales de la "Stack Overflow Developer Survey 2025", una encuesta anual que Stack Overflow realiza a la comunidad de desarrolladores sobre su perfil, trabajo, tecnologías y uso de herramientas de IA.**

## Visión General
* **Dimensiones**
* **Tipos De Datos**
* **Identificador, Duplicados Y Columnas Sin Valor**

In [ ]:
filas, columnas = df_results.shape
print(f"El Dataset \"results\" tiene {filas:,} filas y {columnas} columnas")
print(f"Memoria: {df_results.memory_usage(deep=True).sum() / 1e6:.1f} MB")

df_results.info(verbose=True, show_counts=True)
print("\n")
tipos = (df_results.dtypes.astype(str).value_counts().rename_axis("Tipo de dato").reset_index(name="Cantidad de columnas"))
tipos

print("ResponseId es único?: ", df_results['ResponseId'].is_unique)
print("Filas duplicadas:", df_results.duplicated().sum())
print("Columnas con un solo valor distinto:",[c for c in df_results.columns if df_results[c].nunique(dropna=True) <= 1])

## Interpretación

**El dataset `results` contiene 49.191 respuestas (filas) y 172 variables (columnas), con un uso de memoria de aproximadamente 375 MB.**

- **Tipos de datos: 119 columnas son de texto (`str`), 52 son numéricas de punto flotante (`float64`) y 1 es entera (`int64`, el identificador `ResponseId`). Las numéricas son `float` y no `int` porque contienen valores nulos.**
- **Identificador: `ResponseId` es único, por lo que cada fila representa una respuesta distinta.**
- **Duplicados: no hay filas duplicadas.**
- **Columnas sin información: no hay columnas constantes ni vacías.**

## Previo A La Caracterización De Atributos...

**Falta de contexto en los nombres abreviados: La base `results.zip` utiliza códigos abreviados como nombres de columna (por ejemplo, `AIAcc`, `TechEndorse_1`, `JobSatPoints_3`). Sin el diccionario de datos, es imposible saber la naturaleza exacta de la pregunta o qué representa la respuesta.**

**Determinación precisa del tipo de variable: Conocer el enunciado exacto y la estructura original de la pregunta en `schema.csv` permite definir objetivamente si un atributo es cualitativo ordinal, nominal, o cuantitativo continuo, evitando clasificaciones incorrectas basadas únicamente en el tipo de dato de Pandas (float64 o object).**

**Insumo para las etapas de Limpieza e Hipótesis: Saber qué variables corresponden a preguntas obligatorias, opcionales o condicionales (matriztadas) justifica el porcentaje de valores nulos observado y guía las decisiones de imputación o filtrado para el planteo de hipótesis.**



## Vinculacion Entre `results` Y `schema`

**Para determinar qué atributo de `schema.csv` actúa como clave de diccionario para los nombres de las columnas de `results.csv`, se realizó una comprobación evaluando la intersección entre las 172 columnas de la base de datos de respuestas y los valores únicos de cada campo de la base de esquemas**

In [ ]:

# Comprobación de coincidencia entre las columnas de "results" y las columnas del "schema"
cols_results = set(df_results.columns)

for col in df_schema.columns:
    valores_schema = set(df_schema[col].dropna().astype(str))
    coincidencias = cols_results.intersection(valores_schema)
    print(f"Coincidencias de las columnas de 'results' con df_schema['{col}']: {len(coincidencias)} de {len(cols_results)}")


# Identificación de columnas de results que no coinciden directamente con qname
cols_no_coincidentes = sorted(list(set(df_results.columns) - set(df_schema["qname"])))


print(f"\nTotal de columnas sin relación directa: {len(cols_no_coincidentes)}\n")
df_cols_huerfanas = pd.DataFrame(cols_no_coincidentes, columns=["Columna sin relación directa en qname"])
df_cols_huerfanas

## ¿Qué Hacemos En Este Código?
1. **Verificación de coincidencia de campos:**
   * Recorremos todas las columnas del diccionario (`df_schema`) y calculamos la intersección entre los nombres de columna del dataset principal (`df_results`) y los valores presentes en cada campo del esquema.
2. **Aislamiento de variables sin relación directa:**
   * Identificamos 46 columnas de `results.csv` cuyo nombre no coincide exactamente con ninguna entrada clave (`qname`) de `schema.csv`.
   * Construimos una tabla explicativa (`df_cols_huerfanas`) con estas variables no vinculadas directamente.

## ¿Para Qué Sirve?
1. **Identificar la clave primaria del esquema:** 
   * Permite confirmar que la columna `qname` en el esquema es el identificador clave que da nombre a la mayoría de las variables de la encuesta (logrando 126 coincidencias directas sobre 172).
2. **Detectar variables desglosadas y derivadas:** 
   * Permite aislar aquellas columnas que provienen de preguntas compuestas (como matrices Likert o selección múltiple de tecnologías con sufijos como `HaveWorkedWith` o `WantToWorkWith`), así como identificadores técnicos (`ResponseId`) y variables calculadas (`ConvertedCompYearly`).
3. **Base para la caracterización completa:** 
   * Constituye el paso previo necesario para aplicar algoritmos de extracción de raíz y asignar a cada columna del dataset su descripción, tipo de pregunta y módulo temático oficial.

In [ ]:
cols_results = set(df_results.columns)
qnames_schema = set(df_schema["qname"].dropna())
cols_no_coincidentes = sorted(list(cols_results - qnames_schema))

def raiz_mas_larga(col):
    candidatos = [q for q in qnames_schema if col.startswith(q)]  
    return max(candidatos, key=len) if candidatos else None       

def verificar_columna(col):
    if col == "ResponseId":
        return "1. Clave Primaria (PK)", "N/A", "Identificador técnico del encuestado"

    if col == "ConvertedCompYearly":
        return "5. Variable Calculada", "N/A", "Derivada de CompTotal y Currency"

    qname = raiz_mas_larga(col)

    if qname is not None:
        sufijo = col[len(qname):].strip()

        if sufijo in ["HaveWorkedWith", "WantToWorkWith", "Admired"]:
            return "3. Faceta de Tecnología", qname, f"Aspecto: {sufijo}"

        elif sufijo in ["Personal use", "Professional use"]:
            return "4. Ámbito de Uso", qname, f"Ámbito: {sufijo}"

        else:
            return "2. Opción Likert / Matriz", qname, f"Opción concatenada: {sufijo}"

    return "Sin clasificar", "Ninguna", "No se encontró patrón"


resultados = []
for col in cols_no_coincidentes:
    categoria, raiz, detalle = verificar_columna(col)
    resultados.append({
        "Columna en Results": col,
        "Categoría": categoria,
        "Raíz en Schema": raiz,
        "Detalle / Sufijo": detalle
    })

df_verificacion = pd.DataFrame(resultados)

print(f"Total de columnas analizadas: {len(df_verificacion)}")
print("\n===Distribución Por Categoría===")
df_verificacion["Categoría"].value_counts().reset_index()


## ¿Qué Hacemos En Este Código?
1. **Aislamiento de variables sin coincidencia exacta:**
   * Obtenemos la lista de las 46 columnas del dataset de respuestas (`results.csv`) que no tienen un nombre idéntico a ninguna pregunta (`qname`) en el esquema.
2. **Algoritmo de búsqueda de raíz (`raiz_mas_larga`):**
   * Analizamos el nombre de cada columna huérfana para encontrar qué código de pregunta del esquema sirve de prefijo o "pregunta madre".
3. **Clasificación por patrones de sufijo (`verificar_columna`):**
   * Analizamos el texto restante (sufijo) y asignamos cada columna a una de 5 categorías funcionales:
     * **Clave Primaria (PK):** Identificador técnico (`ResponseId`).
     * **Variable Calculada:** Salario estandarizado en USD (`ConvertedCompYearly`).
     * **Faceta de Tecnología:** Desgloses de uso, interés o admiración (`HaveWorkedWith`, `WantToWorkWith`, `Admired`).
     * **Ámbito de Uso:** Separación entre entorno laboral y personal (`Personal use`, `Professional use`).
     * **Opción Likert / Matriz:** Respuestas aplanadas de tablas o matrices de opinión.
4. **Resumen de resultados (`df_verificacion`):**
   * Generamos un resumen con el recuento total de variables pertenecientes a cada una de estas categorías.

## ¿Para Qué Sirve?
1. **Demostrar la trazabilidad del $100\%$ de los datos:** 
   * Probar que las 46 columnas que parecían no tener correspondencia en el diccionario no son errores, sino desgloses de preguntas complejas o variables sintéticas.
2. **Automatizar la asignación de contexto:**
   * Permite conectar automáticamente cada sub-columna con su pregunta de origen en `schema.csv` sin necesidad de hacerlo manualmente.
3. **Insumo para la tabla maestra de variables:**
   * Prepara las bases para construir la tabla unificada `df_variables`, que relaciona cada columna de respuestas con el texto completo de su pregunta y su formato.

In [ ]:
info = df_schema.drop_duplicates("qname")[["qname", "question", "type"]]

df_variables = pd.DataFrame({"ColumnaEnResult": df_results.columns})

df_variables["raiz"] = df_variables["ColumnaEnResult"].apply(
    lambda c: c if c in qnames_schema else raiz_mas_larga(c)
)

df_variables = (df_variables
                .merge(info, left_on="raiz", right_on="qname", how="left")
                .drop(columns="qname"))

df_variables["dtype"] = df_variables["ColumnaEnResult"].map(df_results.dtypes.astype(str))

display(df_variables["type"].value_counts(dropna=False).reset_index())
print(f"\nColumnas sin type: {df_variables.loc[df_variables["type"].isna(), "ColumnaEnResult"].tolist()}")

#df_variables.head(10)
df_schema.columns

## ¿Qué Hacemos En Este Código?

1. **Extracción de metadatos del diccionario (`info`):**
   * Eliminamos duplicados en `df_schema` sobre la columna `qname` para obtener una tabla limpia que relaciona cada pregunta con su texto completo (`question`) y su tipo o formato (`type`).
2. **Creación de la tabla unificada (`df_variables`):**
   * Creamos un DataFrame base con las 172 columnas de `df_results` y asociamos a cada una su código raíz en el esquema (coincidencia directa o prefijo `raiz_mas_larga`).
3. **Enriquecimiento y cruce de datos (`merge`):**
   * Realizamos una unión (*left join*) para incorporar el texto exacto de la pregunta y el tipo de pregunta según el formulario original.
   * Mapeamos el tipo de dato nativo asignado por Pandas (`dtype`: `str`, `float64`, `int64`).
4. **Verificación de cobertura y formatos (`value_counts`):**
   * Visualizamos cuántas columnas corresponden a cada tipo de pregunta (`MC = Multiple Choice`, `TE = Texto Libre`, `RO = Orden de Preferencia`, `Matrix = Tabla de Opciones`).
   * Confirmamos qué columnas quedan sin `type` asociado (`ResponseId` y `ConvertedCompYearly`).

## ¿Para Qué Sirve?
1. **Crear el Diccionario Maestro del proyecto:** 
   * Reúne en un solo lugar toda la metadata técnica y de negocio de las 172 variables del dataset, evitando la necesidad de consultar manualmente `schema.csv` durante el análisis.
2. **Entender el formato de cada pregunta:** 
   * Saber si una variable es de opción múltiple, matriz o texto libre condiciona directamente cómo deben tratarse sus valores nulos, cómo se analizan sus frecuencias y qué técnicas de limpieza o codificación se aplicarán.
3. **Validar las excepciones del cuestionario:** 
   * Permite comprobar que únicamente 2 columnas no poseen un tipo de pregunta asignado, justificando que `ResponseId` es una clave primaria técnica y `ConvertedCompYearly` es una variable calculada/derivada por Stack Overflow.

In [ ]:
def mapear_GrupoOficial(col):
    # Developers (Profile, Education & Experience, Developer roles)
    if col in [
        "ResponseId", "MainBranch", "Age", "Country", "EdLevel", "WorkExp", 
        "YearsCode", "LearnCode", "LearnCodeChoose", "LearnCodeAI", 
        "AILearnHow", "DevType"
    ]:
        return "Developers"
    
    # Technology (Stack tecnológico y herramientas)
    elif any(col.startswith(prefijo) for prefijo in [
        "Language", "Database", "Platform", "Webframe", "DevEnv", 
        "SOTags", "OfficeStack", "CommPlatform", "OpSys"
        ]) or col in ["ToolCountWork", "ToolCountPersonal"]:
        return "Technology"
    
    # AI (Herramientas, modelos, sentimiento y agentes de IA)
    elif col.startswith("AI") or col.startswith("Agent") or "AIAgent" in col or "AITool" in col:
        return "AI"
    
    # Work (Empleo, salario, industria, cultura y satisfacción laboral)
    elif col in [
        "Employment", "EmploymentAddl", "OrgSize", "ICorPM", "RemoteWork", 
        "PurchaseInfluence", "Industry", "Currency", "CompTotal", 
        "ConvertedCompYearly", "JobSat"
    ] or col.startswith("JobSatPoints"):
        return "Work"
    
    # Stack Overflow (Uso, frecuencia y participación en la comunidad)
    elif col.startswith("SO") or col.startswith("SO_"):
        return "Stack Overflow"
    
    # Methodology (Percepción tecnológica, adopción/rechazo de prácticas)
    elif col.startswith("TechEndorse") or col.startswith("TechOppose") or col in ["TechEndorseIntro", "NewRole"]:
        return "Methodology"
    
    return "Sin Clasificar"

df_variables["GrupoOficial"] = df_variables["ColumnaEnResult"].apply(mapear_GrupoOficial)
print(f"Numero De Variables Developers : {(df_variables["GrupoOficial"] == "Developers").sum()}")
print(f"Numero De Variables Technology: {(df_variables["GrupoOficial"] == "Technology").sum()}")
print(f"Numero De Variables AI: {(df_variables["GrupoOficial"] == "AI").sum()}")
print(f"Numero De Variables Work: {(df_variables["GrupoOficial"] == "Work").sum()}")
print(f"Numero De Variables Stack Overflow: {(df_variables["GrupoOficial"] == "Stack Overflow").sum()}")
print(f"Numero De Variables Methodology: {(df_variables["GrupoOficial"] == "Methodology").sum()}")

## ¿Qué Hacemos En Este Código?

1. **Definición de la función de mapeo (`mapear_GrupoOficial`):**
   * Creamos una función basada en condicionales y filtrado por prefijos para clasificar las 172 columnas del dataset en los 6 grupos temáticos oficiales de la encuesta.
2. **Clasificación en 6 grupos:**
   * **Developers:** Perfil demográfico, nivel educativo, experiencia y rol del desarrollador (`Age`, `EdLevel`, `WorkExp`, `YearsCode`, `DevType`, etc.).
   * **Technology:** Lenguajes, bases de datos, plataformas, entornos y herramientas del stack técnico (`Language`, `Database`, `Platform`, `OpSys`, etc.).
   * **AI:** Sentimiento, uso, modelos y agentes de Inteligencia Artificial (`AI...`, `AIAgent...`, `AITool...`).
   * **Work:** Situación laboral, salario, industria, cultura y satisfacción en el trabajo (`Employment`, `CompTotal`, `ConvertedCompYearly`, `RemoteWork`, etc.).
   * **Stack Overflow:** Frecuencia de uso, interacción y participación dentro de la comunidad (`SO...`, `SO_...`).
   * **Methodology:** Percepción, adopción o rechazo de metodologías y prácticas tecnológicas (`TechEndorse...`, `TechOppose...`).
3. **Poblado de la Tabla Maestra:**
   * Aplicamos la función sobre cada fila de `df_variables` para crear la columna `GrupoOficial`.

## ¿Para qué sirve?
1. **Estructurar un Dataset de Alta Dimensionalidad:** 
    * Permite organizar las 172 variables en 6 categorías analíticas claras y manejables.
2. **Facilitar Diagnósticos Agregados:** 
    * Permite calcular métricas a nivel de módulo (como el porcentaje medio de nulos en el módulo de *Inteligencia Artificial* o *Tecnología*) sin tener que analizar las 172 columnas individualmente.
3. **Orientar las Reglas de Limpieza e Hipótesis:** 
    * Ayuda a definir estrategias de preprocesamiento segmentadas (por ejemplo, imputar nulos estructurales en el bloque *Technology/AI* o depurar outliers en el bloque *Work*).

## Diagnóstico de Valores Nulos y Nulos Estructurales


In [ ]:
df_nulos = pd.DataFrame({
    "ColumnaEnResult": df_results.columns,
    "CantidadDeNulos": df_results.isna().sum().values,
    "PorcentajeDeNulos": (df_results.isna().mean() * 100).round(2).values
})

# Agregamos a df_variables la cantidad de nulos y el porcentaje que representan.
if "PorcentajeDeNulos" not in df_variables.columns:
    df_variables = df_variables.merge(df_nulos, on="ColumnaEnResult", how="left")
        
print("===============================")
print(" Resumen De Todos Los Nulos ")
print("================================")
display(df_nulos[df_nulos.columns].head(10))

# Esta parte del código toma la información individual de nulos de cada columna y la agrupa por los 6 módulos temáticos oficiales 

resumen_nulos_grupo = df_variables.groupby("GrupoOficial").agg(
    CantidadDeVariables=("ColumnaEnResult", "count"),
    PromedioDeNulos=("PorcentajeDeNulos", "mean"),
    PorcentajeDeLaVaribleConMenosNulos=("PorcentajeDeNulos", "min"),
    PorcentajeDeLaVaribleConMasNulos=("PorcentajeDeNulos", "max")
).reset_index().sort_values(by="PromedioDeNulos")
resumen_nulos_grupo["PromedioDeNulos"] = resumen_nulos_grupo["PromedioDeNulos"].round(2)

print("================================")
print(" Resumen De Nulos Por Grupos ")
print("================================")
display(resumen_nulos_grupo[resumen_nulos_grupo.columns])


print("=================================================")
print("  DEMOSTRACIÓN EMPÍRICA DE NULOS ESTRUCTURALES ")
print("=================================================")
pd.crosstab(
    df_results["AIModelsChoice"].fillna("Sin Respuesta / NaN"), 
    df_results["AIModelsHaveWorkedWith"].isna().map({True: "Es NaN (Nulo Estructural)", False: "Tiene Respuesta"}),
    margins=True
)



## ¿Qué Hacemos En Este Código?
1. **Cuantificación de nulos a nivel de columna (`df_nulos`):**
   * Calculamos la cantidad absoluta (`CantidadDeNulos`) y el porcentaje (`PorcentajeDeNulos`) de registros vacíos para cada una de las 172 columnas del dataset.
   * Incorporamos estas métricas al diccionario maestro `df_variables` para mantener centralizada la metadata de cada atributo.
2. **Agregación por módulo temático (`resumen_nulos_grupo`):**
   * Agrupamos las variables según su `GrupoOficial` (los 6 módulos temáticos) y calculamos el promedio, mínimo y máximo porcentaje de nulos por módulo.
3. **Demostración empírica mediante matriz de contingencia (`pd.crosstab`):**
   * Cruzamos la respuesta a la pregunta filtro/padre `AIModelsChoice` (*"¿Utilizas modelos de IA?"*) con la presencia o ausencia de respuesta en la pregunta subordinada `AIModelsHaveWorkedWith` (*"¿Con qué modelos has trabajado?"*).

## ¿Para Qué Sirve?
1. **Evaluar la integridad de la información por módulo:** 
    * Permite observar que bloques como *Developers* tienen bajo porcentaje de nulos (~12%), mientras que *Technology* y *AI* superan el 65% en promedio.
2. **Demostrar la existencia de Nulos Estructurales:** 
    * Prueba empíricamente que el 100% de los usuarios que respondieron *"No"* en la pregunta filtro generaron automáticamente un valor `NaN` en las preguntas siguientes de herramientas específicas.
4. **Evitar la eliminación masiva e incorrecta de datos:** 
    * Justifica que los valores faltantes en variables condicionales no son errores de medición ni falta de respuesta arbitraria, sino la representación de la opción **"No aplica / No utiliza"** por diseño de la interfaz web (UI/UX).
5. **Fundamentar las reglas del Punto 2 (Limpieza):** 
    * Establece el criterio técnico para re-categorizar estos `NaN` con la etiqueta `"No aplica / No utiliza"` en lugar de imputar por media/moda o borrar las filas de la muestra.

## Distribución de Variables Cuantitativas y Detección de Outliers


In [ ]:
# 1. Selección de variables cuantitativas principales
cols_cuanti = ["YearsCode", "WorkExp", "ConvertedCompYearly", "CompTotal", "ToolCountWork", "ToolCountPersonal"]

# 2. Tabla de Distribución (Estadísticos Descriptivos + IQR)
df_num_stats = df_results[cols_cuanti].describe().T
df_num_stats["IQR"] = df_num_stats["75%"] - df_num_stats["25%"]
df_num_stats = df_num_stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "IQR"]]
df_num_stats.columns = ["Registros", "Media", "Desvío Std", "Mínimo", "Q1 (25%)", "Mediana (50%)", "Q3 (75%)", "Máximo", "IQR"]

print("=========================================================================================")
print("  DISTRIBUCIÓN Y MÉTRICAS DE VARIABLES CUANTITATIVAS")
print("=========================================================================================\n")
display(df_num_stats.round(2))

# 3. Detección de Outliers e Inconsistencias Lógicas
print("\n=========================================================================================")
print("  DETECCIÓN DE OUTLIERS E INCONSISTENCIAS LÓGICAS")
print("=========================================================================================\n")
print(f"1. WorkExp > YearsCode (Inconsistencia lógica): {(df_results["WorkExp"] > df_results["YearsCode"]).sum():,} casos")
print(f"2. Salario Anual > $2.000.000 USD (Outliers superiores): {(df_results["ConvertedCompYearly"] > 2000000).sum():,} casos")
print(f"3. Salario Anual == $0 USD (Outliers inferiores): {(df_results["ConvertedCompYearly"] == 0).sum():,} casos")
print(f"4. Años programando > 50 años (Valores extremos): {(df_results["YearsCode"] > 50).sum():,} casos")

## ¿Qué Hacemos En Este Código?
1. **Selección de variables cuantitativas clave (`cols_cuanti`):**
   * Filtramos del dataset crudo las 6 variables numéricas principales del estudio: trayectoria profesional (`YearsCode`, `WorkExp`), ingresos (`ConvertedCompYearly`, `CompTotal`) y volumen de herramientas (`ToolCountWork`, `ToolCountPersonal`).
2. **Construcción de la tabla de distribución y dispersión (`df_num_stats`):**
   * Calculamos los estadísticos descriptivos tradicionales mediante `.describe().T` (Registros válidos, Media, Desvío Estándar, Mínimo, Cuartiles y Máximo).
   * Calculamos e incorporamos explícitamente el **Rango Intercuartílico (IQR)** ($Q3 - Q1$) como medida de dispersión robusta y renombramos los encabezados para su lectura ejecutiva.
3. **Auditoría de inconsistencias lógicas y detección de outliers:**
   * Cuantificamos las incoherencias físicas entre variables donde la experiencia laboral supera a los años totales de programación (`WorkExp > YearsCode`).
   * Medimos la presencia de valores atípicos superiores e inferiores en salarios anuales ($> \$2.000.000$ USD o $== \$0$ USD) y años de experiencia extremos ($> 50$ años).

## ¿Para Qué Sirve?
1. **Caracterizar la distribución de las variables numéricas:** 
   * Permite analizar la tendencia central y la dispersión de las variables continuas y discretas del dataset en su estado crudo.
2. **Evaluar el sesgo y fundamentar el uso de métricas robustas:** 
   * Demuestra numéricamente si la Media se encuentra inflada por valores atípicos (como salarios astronómicos), justificando el uso de la Mediana y el IQR como las métricas adecuadas para describir los ingresos y la trayectoria del mercado.
3. **Aislar errores lógicos e inexactitudes de carga:** 
   * Identifica contradicciones físicas imposibles y respuestas tipo *spam* o atípicas introducidas en los campos abiertos del cuestionario web.
4. **Fundamentar las reglas del Punto 2 (Limpieza y Preparación):** 
   * Proporciona la evidencia empírica y los umbrales numéricos concretos para justificar el criterio técnico de filtrado (cuándo eliminar filas incompletas o cuándo re-categorizar valores a `NaN`) antes de contrastar las hipótesis del trabajo.

## Distribución de Variables Cualitativas Principales

In [ ]:
vars_cualitativas = ["MainBranch", "Employment", "EdLevel", "RemoteWork", "Country", "DevType"]

print("=========================================================================================")
print("  DISTRIBUCIÓN DE FRECUENCIAS EN VARIABLES CUALITATIVAS CLAVE ")
print("=========================================================================================\n")

for var in vars_cualitativas:
    if var in df_results.columns:
        print(f"--- Variable: {var} ---")
        conteo = df_results[var].value_counts(dropna=False).head(5)
        pct = (df_results[var].value_counts(dropna=False, normalize=True).head(5) * 100).round(2)
        
        df_freq = pd.DataFrame({
            'Categoría / Respuesta': conteo.index.astype(str),
            'Frecuencia Absoluta': conteo.values,
            'Porcentaje (%)': pct.values
        })
        display(df_freq)
        print("\n")

## ¿Qué Hacemos En Este Código?
1. **Selección de variables cualitativas clave (`vars_cualitativas`):**
   * Filtramos del dataset las 6 variables categóricas principales que describen el perfil demográfico, académico y laboral de los encuestados (`MainBranch`, `Employment`, `EdLevel`, `RemoteWork`, `Country` y `DevType`).
2. **Cálculo de frecuencias absolutas y relativas:**
   * Recorremos iterativamente cada variable calculando el número total de apariciones por respuesta (`value_counts`) y su proporción porcentual respecto al total de encuestados (`normalize=True * 100`).
   * Preservamos los valores nulos (`dropna=False`) para reflejar la participación real sobre la muestra completa sin distorsionar las proporciones.
3. **Generación de tablas resumidas de frecuencia (*Top 5*):**
   * Extraemos las 5 respuestas con mayor volumen para cada atributo (`.head(5)`) y las presentamos en dataframes individuales (`df_freq`) con columnas ejecutivas (*Categoría / Respuesta*, *Frecuencia Absoluta* y *Porcentaje (%)*).

## ¿Para Qué Sirve?
1. **Caracterizar el perfil categórico de la muestra:** 
   * Permite identificar los rasgos dominantes de la comunidad analizada (distribución por país de residencia, nivel educativo alcanzado, especialidad técnica o relación contractual).
2. **Evaluar la concentración y representatividad de subgrupos:** 
   * Facilita medir la adopción real de modalidades de trabajo (remoto, híbrido, presencial) y determinar qué perfiles o regiones concentran la mayor densidad de participantes.
3. **Detectar el peso relativo de los valores nulos:** 
   * Permite visibilizar la proporción de respuestas omitidas en atributos categóricos clave antes de aplicar las transformaciones del dataset.
4. **Fundamentar la recategorización previa al análisis bivariado (Punto 2 y 3):** 
   * Proporciona el sustento empírico para decidir si conviene agrupar categorías minoritarias bajo etiquetas consolidadas (como `"Otros"`) y así evitar tablas de contingencia hiperdispersas al cruzar hipótesis.

## Limpieza Y Preparacion De Datos


In [ ]:
# Trabajaremos sobre df_clean para mantener df_results como respaldo
df_clean = df_results.copy()

print(f"Dimensiones iniciales antes de la limpieza: {df_clean.shape[0]:,} filas y {df_clean.shape[1]} columnas")
mask_inconsistencia_exp = df_clean["WorkExp"] > df_clean["YearsCode"]
casos_inconsistentes = mask_inconsistencia_exp.sum()

# Filtrado: conservamos solo los registros coherentes o con valores NaN
df_clean = df_clean[~mask_inconsistencia_exp | df_clean["WorkExp"].isna() | df_clean["YearsCode"].isna()]

print(f"Registros eliminados por WorkExp > YearsCode: {casos_inconsistentes:,}")
print(f"Filas restantes: {df_clean.shape[0]:,}")
# 1. Salario Anual Convertido (ConvertedCompYearly): Rango realista [100 USD - 2.000.000 USD]
mask_salario_outlier = (
    (df_clean["ConvertedCompYearly"] < 100) | 
    (df_clean["ConvertedCompYearly"] > 2000000)
)

# Limite máximo biológicamente razonable para años de trabajo (ejemplo: max 50 o 60 años)
MAX_WORK_EXP = 50

# Filtrar valores mayores al umbral manteniendo los NaN
df_clean = df_clean[(df_clean["WorkExp"] <= MAX_WORK_EXP) | df_clean["WorkExp"].isna()].copy()

# Convertimos los salarios outliers a NaN para no perder toda la fila del encuestado
salarios_afectados = mask_salario_outlier.sum()
df_clean.loc[mask_salario_outlier, "ConvertedCompYearly"] = None

# 2. Control de años de programación inverosímiles (> 60 años)
mask_yearscode_extremo = df_clean["YearsCode"] > 60
df_clean.loc[mask_yearscode_extremo, "YearsCode"] = None

print(f"Salarios atípicos/spam re-categorizados como NaN: {salarios_afectados:,}")
print(f"Registros de YearsCode > 60 re-categorizados como NaN: {mask_yearscode_extremo.sum():,}")
# Columnas condicionales clave identificadas en el EDA
cols_tecnologia_ai = [c for c in df_clean.columns if any(c.startswith(p) for p in [
    "Language", "Database", "Platform", "Webframe", "DevEnvs", "AIModels", "AITool", "AIAgent"
])]

# Imputación semántica de nulos condicionales
for col in cols_tecnologia_ai:
    if df_clean[col].dtype == 'object' or df_clean[col].dtype == 'str':
        df_clean[col] = df_clean[col].fillna("No aplica / No utiliza")

print(f"Se imputó la categoría 'No aplica / No utiliza' en {len(cols_tecnologia_ai)} columnas condicionales.")
filas_orig = df_results.shape[0]
filas_fin = df_clean.shape[0]
retencion_pct = (filas_fin / filas_orig) * 100

print("=========================================================")
print("  BALANCE GENERAL DE LIMPIEZA Y PREPARACIÓN DE DATOS")
print("=========================================================")
print(f"Filas originales:         {filas_orig:,}")
print(f"Filas limpias finales:    {filas_fin:,}")
print(f"Filas eliminadas:         {filas_orig - filas_fin:,} (Inconsistencias severas)")
print(f"Porcentaje de retención:  {retencion_pct:.2f}% de la muestra")
print("=========================================================")

#df_clean.to_pickle("df_clean.pkl")



## ¿Qué Hacemos En Este Código?
1. **Creación de la copia de resguardo (`df_clean`):**
   * Generamos una copia exacta e independiente de `df_results` para aplicar las reglas de depuración sobre `df_clean` sin alterar la base cruda original.
2. **Depuración de inconsistencias lógicas físicas y límites de experiencia:**
   * Eliminamos las observaciones donde la experiencia laboral declarada supera a los años totales escribiendo código (`WorkExp > YearsCode`).
   * Aplicamos un filtro de corte sobre valores de experiencia laboral profesional que exceden los 50 años (`WorkExp > 50`).
3. **Tratamiento defensivo de valores atípicos (Sin descarte de encuestados):**
   * Identificamos salarios fuera del rango realista ($< \$100$ USD o $> \$2.000.000$ USD) y años de programación inverosímiles ($> 60$ años), reemplazando únicamente dichos valores por `NaN` en lugar de borrar la fila completa.
4. **Imputación semántica de nulos estructurales:**
   * Recorremos las columnas condicionales de tecnología e Inteligencia Artificial (`Language`, `Database`, `Platform`, `AIModels`, etc.) reemplazando los nulos por la etiqueta explícita `"No aplica / No utiliza"`.
5. **Auditoría del balance general y tasa de retención:**
   * Cuantificamos la cantidad de registros eliminados e imprimimos el porcentaje final de conservación de la muestra respectiva.

## ¿Para Qué Sirve?
1. **Asegurar la coherencia interna de los datos:** 
   * Remueve incongruencias físicas imposibles que sesgarían las relaciones analíticas entre trayectoria y perfil profesional.
2. **Preservar el tamaño muestral reduciendo el sesgo por descarte:** 
   * Al re-categorizar los salarios o datos atípicos como `NaN` en lugar de borrar al usuario completo, evitamos perder información válida del resto de sus respuestas.
3. **Dar valor semántico a los saltos lógicos del cuestionario:** 
   * Transforma las ausencias derivadas de la arquitectura del formulario (UI/UX) en la categoría analítica `"No aplica / No utiliza"`, diferenciando el desinterés/no uso de la falta de respuesta arbitraria.
4. **Validar metodológicamente la calidad del dataset final:** 
   * Proporciona y documenta el balance de retención muestral para el informe final, garantizando que el filtrado preservó la representatividad estadística exigida para las pruebas de hipótesis.

## Distribucion De Las Variables Post Limpieza

In [ ]:
import sweetviz as sv
from IPython.display import IFrame, display

# 1. Muestreo aleatorio representativo (5.000 registros para estabilidad)
if len(df_clean) > 5000:
    df_sweetviz = df_clean.sample(n=5000, random_state=42).copy()
else:
    df_sweetviz = df_clean.copy()

# 2. Selección del subconjunto representativo por módulos
cols_sweetviz = [
    # Cuantitativas (Experiencia, Compensación, Herramientas)
    "YearsCode", "WorkExp", "ConvertedCompYearly", "CompTotal", "ToolCountWork", "ToolCountPersonal",
    # Cualitativas (Perfil, Empleo, Educación, Organización)
    "MainBranch", "Employment", "EdLevel", "RemoteWork", "Country", "DevType"
]

# Filtrar columnas existentes
cols_validas = [c for c in cols_sweetviz if c in df_sweetviz.columns]
df_sweetviz = df_sweetviz[cols_validas]

# 3. Tratamiento de nulos para categorización en Sweetviz
for col in df_sweetviz.select_dtypes(include=['object']).columns:
    df_sweetviz[col] = df_sweetviz[col].fillna("No aplica / No utiliza").astype(str)

# 4. Generación y exportación del reporte interactivo
reporte = sv.analyze(
    source=df_sweetviz,
    pairwise_analysis="off"
)

reporte.show_notebook()

# Hipotesis Planteadas Por El Grupo

## Hipótesis Univariada: Experiencia Laboral Profesional en Argentina

"Los desarrolladores que responden la encuesta desde Argentina tienen, en promedio, más de 12 años de experiencia laboral profesional".

**Población de estudio:** Desarrolladores encuestados residentes en Argentina con registro válido en la variable de experiencia laboral (`WorkExp`).

**Hipótesis Nula ($H_0$):** Los desarrolladores en Argentina tienen una experiencia laboral media menor o igual a 12 años ($\mu \le 12$).

**Hipótesis Alternativa ($H_1$):** Los desarrolladores en Argentina tienen una experiencia laboral media estrictamente superior a 12 años ($\mu > 12$).

**Atributos involucrados:**
- `WorkExp`: Años de experiencia laboral profesional (cuantitativa discreta/continua).
- `Country`: País de residencia del encuestado (cualitativa nominal), utilizado como filtro para acotar la muestra a Argentina.

**Justificación y Marco Teórico:**
El umbral de 12 años permite distinguir a profesionales con una trayectoria consolidada dentro de la industria tecnológica de perfiles *junior* o intermedios. Dado que la media global de experiencia en la encuesta de Stack Overflow ronda los 12,7 años, evaluar si la comunidad técnica argentina supera o iguala este estándar internacional brinda un indicador sobre la madurez del perfil profesional local.

In [ ]:
# 1. Filtrar por Argentina
df_arg = df_clean[df_clean["Country"] == "Argentina"].copy()


# 3. Configurar y crear el gráfico
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Graficar la distribución de los años de experiencia
sns.histplot(data=df_arg, x="WorkExp", bins=20, color="blue", edgecolor="black")

# Configurar títulos y etiquetas
plt.title("Distribución de Años de Experiencia (WorkExp) en Argentina", fontsize=14, pad=15)
plt.xlabel("Años de Experiencia Profesional", fontsize=12)
plt.ylabel("Cantidad de Desarrolladores", fontsize=12)

# Mostrar la leyenda para identificar las líneas
plt.legend(fontsize=11)

# Ajustar el diseño y mostrar
plt.tight_layout()
plt.show()

#Validacion de hipotesis

# Grafico qqplot para analizar la normalidad de forma visual
workexp = df_arg["WorkExp"].dropna()

plt.figure(figsize=(8, 5))
stats.probplot(workexp, dist="norm", plot=plt)

plt.title("Gráfico Q-Q de WorkExp (Argentina)", fontsize=14)
plt.xlabel("Cuantiles Teóricos (Distribución Normal)", fontsize=12)
plt.ylabel("Valores Reales de WorkExp", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

#Analizamos normalidad con el test de Shapiro-wilk
estadistico, p_valor = stats.shapiro(workexp)

print("--- Test de Shapiro-Wilk ---")
print(f"Estadístico: {estadistico}")
print(f"Valor p: {p_valor}")

if p_valor > 0.05:
    print("Conclusión: No se rechaza la normalidad")
else:
    print("Conclusión: Se rechaza la normalidad (Los datos NO siguen una distribución normal).")

### ¿Qué hacemos en este paso?
1. **Inspección de Frecuencias y Cuantiles:** Generamos el histograma empírico de `WorkExp` y el gráfico de probabilidad normal ($Q\text{-}Q\text{ Plot}$) para la submuestra de Argentina.
2. **Prueba Formal de Normalidad:** Aplicamos el Test de Shapiro-Wilk (`stats.shapiro`) para contrastar la hipótesis de normalidad poblacional.

### ¿Por qué lo hacemos?
- **Diagnóstico de Asimetría:** Tanto la inspección gráfica ($Q\text{-}Q\text{ Plot}$) como el valor de $p < 0,05$ obtenido en la prueba de Shapiro-Wilk confirman que la variable `WorkExp` no sigue una distribución normal estricta, exhibiendo asimetría y valores atípicos típicos del seniority profesional.
- **Robustez por Teorema del Límite Central (TLC):** A pesar de la falta de normalidad en la variable original, dado que el tamaño muestral para Argentina supera ampliamente el umbral de $n \ge 30$ observaciones, el Teorema del Límite Central asegura la normalidad asintótica de la media muestral ($\bar{X}$), garantizando la validez formal del test $t$ de Student.

In [ ]:
# 2. Definir los parámetros del test
valor_hipotesis = 12


# 3. Ejecutar el T-test de una muestra
# alternative='greater' indica que probamos si la media poblacional es MAYOR a 12
t_stat, p_value = stats.ttest_1samp(workexp, popmean=valor_hipotesis, alternative='greater')

# 4. Mostrar los resultados numéricos
print(f"Media muestral: {workexp.mean():.2f} años")
print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor p (p-value): {p_value:.4e}\n")

# 5. Interpretar el resultado estadístico
if p_value < 0.05:
    print("Conclusión: Rechazamos la hipótesis nula.")
    print("Hay evidencia estadística suficiente para afirmar que los desarrolladores argentinos tienen, en promedio, más de 12 años de experiencia.")
else:
    print("Conclusión: No podemos rechazar la hipótesis nula.")
    print("No hay evidencia estadística suficiente para afirmar que la media supera los 12 años.")
    

### ¿Qué hacemos en este paso?
1. **Depuración de Faltantes:** Filtramos las observaciones con datos nulos en la variable `WorkExp` dentro de la submuestra de Argentina.
2. **Prueba Paramétrica de Una Cola:** Aplicamos la prueba $t$ de Student para una muestra evaluando el parámetro $\mu_0 = 12$ con la opción `alternative='greater'`.
3. **Evaluación de Significancia:** Contrastamos el $p\text{-valor}$ resultante frente al nivel de significancia prefijado $\alpha = 0,05$.

### ¿Por qué lo hacemos?
- **Evaluación de Direccionalidad:** Dado que la hipótesis alternativa ($H_1$) plantea específicamente un valor promedio *superior* a 12 años, corresponde aplicar un test unilateral hacia la cola derecha de la distribución.


A partir de la muestra depurada de desarrolladores residentes en Argentina ($n > 30$), la prueba **$t$ de Student para una muestra** arrojó los siguientes resultados inferenciales:

- **Media muestral ($\bar{X}$):** 14,07 años de experiencia profesional.
- **Estadístico $T$:** 2,8066
- **Valor $p$ ($p\text{-valor}$):** $0,0028$ ($2,78 \times 10^{-3}$)

Dado que el $p\text{-valor}$ obtenido ($0,0028$) es marcadamente inferior al nivel de significancia prefijado ($\alpha = 0,05$), existe evidencia empírica suficiente para desestimar la hipótesis nula.

Por lo tanto, **SE RECHAZA LA HIPÓTESIS NULA ($H_0$)** y se valida la **HIPÓTESIS ALTERNATIVA ($H_1$)**, concluyendo que los desarrolladores que responden la encuesta desde Argentina tienen, en promedio, más de 12 años de experiencia laboral profesional ($\mu > 12$).


## Hipótesis Bivariada: Impacto del Uso de Herramientas de IA en la Compensación Salarial

**"Los desarrolladores independientes que utilizan herramientas de IA tienen una compensación salarial mediana superior a aquellos que no las utilizan".**

**Población de estudio:** Desarrolladores independientes / *freelancers* de la muestra `df_clean` con registros salariales válidos.

**Hipótesis Nula ($H_0$):** La compensación salarial mediana de los desarrolladores independientes que utilizan herramientas de IA es menor o igual a la de aquellos que no las utilizan ($\text{Mediana}_{\text{IA}} \le \text{Mediana}_{\text{No IA}}$).

**Hipótesis Alternativa ($H_1$):** La compensación salarial mediana de los desarrolladores independientes que utilizan herramientas de IA es estrictamente mayor a la de aquellos que no las utilizan ($\text{Mediana}_{\text{IA}} > \text{Mediana}_{\text{No IA}}$).

**Atributos involucrados:**
- `uso_ia`: Condición de adopción de herramientas de IA (*Usa IA* vs. *No usa IA*, cualitativa dicotómica/binaria).
- `ConvertedCompYearly`: Compensación total anual estandarizada en dólares (cuantitativa continua).
- `Employment`: Modalidad de empleo (cualitativa nominal), utilizada como filtro para acotar la muestra al segmento independiente.

**Justificación y Marco Teórico:**
La adopción de IA generativa promete incrementos sustanciales en la productividad individual. En trabajadores independientes y *freelancers*, donde los ingresos están vinculados de forma directa a la velocidad y volumen de entregables, evaluar si el uso de IA genera un diferencial salarial significativo permite cuantificar el impacto económico real de estas tecnologías en la facturación profesional.


In [ ]:
# Me quedo con los desarrolladores independientes
empleo = ["Independent contractor, freelancer, or self-employed"]
df_emp = df_clean[df_clean["Employment"].isin(empleo)].copy()

condiciones = [df_emp["AISelect"].str.startswith("Yes", na=False),df_emp["AISelect"].str.startswith("No", na=False),]
opciones = ["Usa IA", "No usa IA"]

df_emp["uso_ia"] = np.select(condiciones, opciones, default="Sin respuesta")

salarioAnual = "ConvertedCompYearly"

#Elimino los nan de los salrios
df_emp = df_emp.dropna(subset=[salarioAnual]).copy()

# Filtrar registros válidos excluyendo 'Sin respuesta'
df_plot = df_emp[
    (df_emp["uso_ia"].isin(["Usa IA", "No usa IA"]))].copy()

#Gráfico
plt.figure(figsize=(9, 6))

# Boxplot base para ver la dispersión y la mediana
sns.boxplot(
    data=df_plot,
    x="uso_ia",
    y=salarioAnual,
    palette=["#B0BEC5", "#1E88E5"],
    width=0.35,
    showfliers=False,
)

plt.title(
    f"Compensación Salarial según Uso de IA)",
    fontsize=12,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Uso Activo de IA", fontsize=10)
plt.ylabel("Compensación Salarial Anual (USD)", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
grupo_ia = df_emp[df_emp["uso_ia"] == "Usa IA"][salarioAnual]
grupo_no_ia = df_emp[df_emp["uso_ia"] == "No usa IA"][salarioAnual]

# SUPUESTO 1: NORMALIDAD

# A. Análisis Gráfico: Q-Q Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

stats.probplot(grupo_ia, dist="norm", plot=ax[0])
ax[0].set_title("Q-Q Plot: Desarrolladores que Usan IA")

stats.probplot(grupo_no_ia, dist="norm", plot=ax[1])
ax[1].set_title("Q-Q Plot: Desarrolladores que NO Usan IA")

plt.tight_layout()
plt.show()

### ¿Qué hacemos en este paso?
1. **Separación de Muestras:** Dividimos a los desarrolladores independientes en dos subgrupos analíticos: `grupo_ia` (*Usa IA*) y `grupo_no_ia` (*No usa IA*).
2. **Inspección de Forma y Simetría:** Construimos los gráficos de probabilidad normal ($Q\text{-}Q\text{ Plot}$) para la compensación salarial en ambos contingentes.
3. **Prueba Formal de Normalidad:** Aplicamos el Test de Shapiro-Wilk (`stats.shapiro`) de forma independiente sobre cada submuestra salarial.

### ¿Por qué lo hacemos?
- **Verificación de Requisitos Paramétricos:** La variable salarial `ConvertedCompYearly` suele presentar una marcada asimetría positiva y presencia de valores atípicos severos.
- **Diagnóstico del Test :** Si los gráficos de probabilidad se desvían de la recta teórica y el test de Shapiro-Wilk arroja un $p < 0,05$ en cualquiera de los grupos, el supuesto de normalidad requerida por la prueba $t$ de Student de dos muestras independientes queda descartado, justificando la necesidad de aplicar una alternativa no paramétrica.


In [ ]:
stat, p_valor = stats.mannwhitneyu(x=grupo_ia, y=grupo_no_ia, alternative='greater')

# 3. Mostrar los resultados
print("--- Test de Mann-Whitney U ---")
print(f"Estadístico U: {stat}")
print(f"p-valor: {p_valor:.4e}\n")

if p_valor < 0.05:
    print("Conclusión: Rechazamos la Hipótesis Nula (H0).")
    print("Existe evidencia estadística significativa para afirmar que los desarrolladores que utilizan herramientas de IA tienen una compensación salarial mayor.")
else:
    print("Conclusión: No podemos rechazar la Hipótesis Nula (H0).")
    print("No existe evidencia estadística suficiente para afirmar que el uso de herramientas de IA impacte positivamente en el salario.")

### ¿Qué hacemos en este paso?
1. **Prueba No Paramétrica para Muestras Independientes:** Aplicamos el test U de Mann-Whitney (`stats.mannwhitneyu`) comparando la distribución de salarios de `grupo_ia` contra `grupo_no_ia`.
2. **Especificación de Unilateralidad:** Configuramos el parámetro `alternative='greater'` para evaluar direccionalmente si la mediana del grupo que utiliza IA domina estocásticamente a la del grupo no usuario.
3. **Evaluación de Significancia:** Contrastamos el $p\text{-valor}$ resultante con el nivel de significancia prefijado $\alpha = 0,05$.

### ¿Por qué lo hacemos?
- **Adecuación Metodológica:** Al trabajar con distribuciones no normales y con presencia de valores atípicos salariales, el test U de Mann-Whitney es el estándar más robusto. Compara los rangos ordinales de las observaciones sin verse distorsionado por los valores extremos propios de la compensación en la industria IT.

Dado que el $p\text{-valor}$ obtenido ($0,5733$) es marcadamente **superior** al nivel de significancia prefijado ($\alpha = 0,05$), no existe evidencia empírica suficiente para desestimar la hipótesis nula.

Por lo tanto, **NO SE RECHAZA LA HIPÓTESIS NULA ($H_0$)**, concluyendo que no existe evidencia estadística suficiente para afirmar que los desarrolladores independientes que utilizan herramientas de IA perciban una compensación salarial mediana mayor que aquellos que no las utilizan.


# Multivariada: Identificación de Arquetipos de Desarrolladores según Trayectoria y Adopción Tecnológica

**Población de estudio:** Desarrolladores.

**Hipótesis Nula ($H_0$):** No existen arquetipos de comportamiento bien diferenciados ni estadísticamente significativos en la muestra; la variación conjunta de la trayectoria profesional (`YearsCode`, `WorkExp`) y el volumen de herramientas utilizadas (`ToolCountWork`, `ToolCountPersonal`) se distribuye de forma homogénea en el espacio multidimensional.

**Hipótesis Alternativa ($H_1$):** Los desarrolladores de la muestra se agrupan en arquetipos de comportamiento bien diferenciados y estadísticamente significativos, definidos de manera conjunta por su trayectoria profesional (`YearsCode`, `WorkExp`), el volumen de tecnologías utilizadas en el ámbito laboral (`ToolCountWork`) y el nivel de experimentación técnica personal (`ToolCountPersonal`).

**Atributos involucrados:**
- `YearsCode`: Años totales programando (cuantitativa continua).
- `WorkExp`: Años de experiencia laboral profesional (cuantitativa continua).
- `ToolCountWork`: Cantidad de herramientas/tecnologías utilizadas en el trabajo (cuantitativa continua/discreta).
- `ToolCountPersonal`: Cantidad de herramientas/tecnologías experimentadas a nivel personal (cuantitativa continua/discreta).

El ecosistema de desarrollo de software abarca perfiles heterogéneos: profesionales con extensa trayectoria orientados a stacks acotados, perfiles de rápida exploración personal de nuevas tecnologías (*early adopters*) y veteranos que gestionan amplios stacks corporativos. Demostrar la presencia de clusters estadísticamente significativos mediante aprendizaje no supervisado permite confirmar la existencia de arquetipos latentes dentro de la comunidad técnica.

In [ ]:
# 1. Selección de variables y limpieza de nulos
cols_multivariada = ["YearsCode", "WorkExp", "ToolCountWork", "ToolCountPersonal"]
df_cluster_clean = df_clean[cols_multivariada].dropna().copy()

# 2. Filtrado estricto de outliers (Descartamos recuentos imposibles > 50)
df_cluster_clean = df_cluster_clean[
    (df_cluster_clean["ToolCountWork"] <= 50) & 
    (df_cluster_clean["ToolCountPersonal"] <= 50)
]

# 3. Estandarización de los datos depurados
scaler = StandardScaler()
X_scaled_clean = scaler.fit_transform(df_cluster_clean[cols_multivariada])

print(f"Tamaño de la muestra tras limpiar outliers: {len(df_cluster_clean)} registros")

### ¿Qué hacemos en este paso?
1. **Selección de Atributos:** Filtramos el DataFrame para conservar únicamente las variables cuantitativas de trayectoria (`YearsCode`, `WorkExp`) y adopción de herramientas (`ToolCountWork`, `ToolCountPersonal`), descartando registros con datos faltantes.
2. **Depuración de Outliers Extremos:** Eliminamos las observaciones que registran conteos de herramientas superiores a 50 (tanto en ámbito laboral como personal), considerándolos valores implausibles o ruidosos.
3. **Escalado de Variables (`StandardScaler`):** Transformamos las cuatro variables para que tengan media $\mu = 0$ y desviación estándar $\sigma = 1$.

### ¿Por qué lo hacemos?
- **Sensibilidad de K-Means a la Escala:** Los algoritmos de agrupamiento basados en la distancia euclidiana atribuyen implícitamente mayor peso a las variables con rangos o varianzas más amplios (como los años de experiencia) por sobre variables de menor magnitud (como la cantidad de herramientas). La estandarización equipara la importancia de todos los atributos en el espacio $4D$.
- **Protección de Centroides:** Los algoritmos de aprendizaje no supervisado son altamente sensibles a valores extremos. Remover outliers imposibles evita que los centroides resultantes se desvíen por anomalías de carga en la encuesta.

In [ ]:
# 4. Búsqueda del K óptimo sin distorsión de outliers
k_range = range(2, 7)
inercias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled_clean)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled_clean, labels))

# 5. Gráficos de evaluación
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(k_range, inercias, marker='o', color='b')
axes[0].set_title('Método del Codo (Datos Limpios)')
axes[0].set_xlabel('Cantidad de Clusters (k)')
axes[0].set_ylabel('Inercia')

axes[1].plot(k_range, silhouettes, marker='s', color='g')
axes[1].set_title('Coeficiente de Silueta (Datos Limpios)')
axes[1].set_xlabel('Cantidad de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

### ¿Qué hacemos en este paso?
Evaluamos el comportamiento de la segmentación para un rango de $k \in [2, 7]$ utilizando dos métricas complementarias:
- **Método del Codo (Inercia intra-cluster):** Mide la suma de las distancias al cuadrado de cada punto a su centroide asignado.
- **Coeficiente de Silueta (`Silhouette Score`):** Mide simultáneamente la cohesión interna de cada cluster y su separación respecto a los clusters vecinos.

### ¿Por qué lo hacemos?
- **Evitar Arbitrariedad:** La elección de la cantidad de arquetipos no debe ser un supuesto a priori, sino una decisión fundamentada en el comportamiento geométrico de los datos.
- **Balance Cohesión vs. Separabilidad:** Buscamos el valor de $k$ donde la reducción de inercia muestre un punto de inflexión ("codo") y el Coeficiente de Silueta alcance un máximo global o relativo que garantice clusters nítidos.

In [ ]:
# 6. Entrenar Modelo Final (Asumimos k=3 como el más representativo)
k_optimo = 3 
kmeans_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df_cluster_clean["Cluster"] = kmeans_final.fit_predict(X_scaled_clean)

# 7. Reducción de Dimensionalidad (PCA 2D)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_clean)
df_cluster_clean["PCA1"] = X_pca[:, 0]
df_cluster_clean["PCA2"] = X_pca[:, 1]

# 8. Gráfico de dispersión
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x="PCA1", y="PCA2", 
    hue="Cluster", 
    palette="viridis", 
    data=df_cluster_clean, 
    alpha=0.6
)
plt.title(f"Proyección PCA de los {k_optimo} Arquetipos (Sin Outliers)")
plt.show()

### ¿Qué hacemos en este paso?
1. **Modelado K-Means:** Entrenamos el algoritmo con $k = 3$ clusters e hiperparámetro `random_state=42` para garantizar reproducibilidad.
2. **Reducción PCA:** Aplicamos Análisis de Componentes Principales (`PCA`) para proyectar el espacio $4D$ original en 2 componentes principales (`PCA1` y `PCA2`).
3. **Visualización Espacial:** Graficamos un diagrama de dispersión 2D coloreando las observaciones según su pertenencia a cada cluster.

### ¿Por qué lo hacemos?
- **Visualización en 2D:** Como la mente humana no puede inspeccionar de forma directa un espacio de 4 dimensiones, el PCA condensa la mayor parte de la varianza total en dos dimensiones cartesianas para evaluar visualmente el grado de separación y solapamiento entre arquetipos.

In [ ]:
mapa_arquetipos = {
    0: "Veteranos de Stack Especializado",
    1: "Perfil Pragmático Estándar",
    2: "Early Adopters Exploradores"
}
df_cluster_clean["Nombre_Arquetipo"] = df_cluster_clean["Cluster"].map(mapa_arquetipos)
# ==================================================

# 9. Tabla de Centroides : Agrupamos por "Nombre_Arquetipo")
centroides = df_cluster_clean.groupby("Nombre_Arquetipo")[cols_multivariada].median()
centroides["Tamaño_Muestra"] = df_cluster_clean["Nombre_Arquetipo"].value_counts()
print("=== PERFIL DE LOS ARQUETIPOS (MEDIANAS) ===")
display(centroides)

# 10. Validación Estadística (Kruskal-Wallis)
print("\n=== VALIDACIÓN ESTADÍSTICA (KRUSKAL-WALLIS) ===")
p_valores = [] 
for col in cols_multivariada:
    # Mantenemos el filtro numérico ("Cluster" == i) para que el loop de k_optimo funcione perfecto
    grupos = [df_cluster_clean[df_cluster_clean["Cluster"] == i][col] for i in range(k_optimo)]
    stat, p_val = kruskal(*grupos)
    p_valores.append(p_val) 
    print(f"Variable '{col}': H-Stat = {stat:.2f} | p-valor = {p_val:.4e}")
    
# 11. Veredicto Final Dinámico (H0 vs H1)
print("\n=== CONCLUSIÓN FORMAL DE LA HIPÓTESIS ===")

if all(p < 0.05 for p in p_valores):
    print("Veredicto: RECHAZAMOS la Hipótesis Nula (H0).")
    print("Dado que todas las variables evaluadas presentan diferencias estadísticamente significativas (p < 0.05),")
    print("podemos afirmar que la Hipótesis Alternativa (H1) es CORRECTA:")
    print("Los desarrolladores de la muestra se agrupan en arquetipos de comportamiento bien diferenciados,")
    print("definidos conjuntamente por su trayectoria profesional y nivel de experimentación técnica.")
else:
    print("Veredicto: NO PODEMOS RECHAZAR la Hipótesis Nula (H0).")
    print("Dado que no todas las variables presentan diferencias significativas, la hipótesis planteada es INCORRECTA.")
    print("No existe evidencia empírica suficiente para separar a la comunidad en arquetipos claramente diferenciados.")

### ¿Qué hacemos en este paso?
1. **Asignación Semántica:** Mapeamos las etiquetas numéricas (`0`, `1`, `2`) a nombres conceptuales representativos de la comunidad técnica.
2. **Cálculo de Centroides:** Agrupamos los datos por arquetipo y calculamos la **mediana** de cada variable en sus unidades originales.
3. **Prueba de Hipótesis No Paramétrica:** Aplicamos el Test de Kruskal-Wallis sobre cada variable en los tres clusters para corroborar significancia estadística ($p < 0.05$).
4. **Veredicto Automático:** Evaluamos si existe suficiente evidencia empírica para rechazar la Hipótesis Nula ($H_0$).

### ¿Por qué lo hacemos?
- **Interpretabilidad de Negocio / Dominio:** Los algoritmos únicamente asignan números; bautizar y caracterizar los grupos permite traducir la matemática a perfiles reales del ecosistema software.
- **Uso de Medianas:** Las medianas son más robustas que las medias frente a la asimetría propia de los conteos de herramientas y años de experiencia.
- **Rigor Estadístico:** El algoritmo K-Means *siempre* dividirá los datos en $k$ grupos, incluso si son aleatorios. El Test de Kruskal-Wallis valida científicamente que las distribuciones subyacentes entre los grupos difieren de forma verdaderamente significativa y no por azar.

### Descripción de los Arquetipos Identificados
A partir de los centroides analizados, la comunidad de desarrolladores se agrupa en tres arquetipos de comportamiento bien consolidados:

1. **Veteranos de Stack Especializado (Cluster 0):** Desarrolladores de extensa trayectoria (mediana de 30 años programando y 25 de experiencia laboral), enfocados en un ecosistema de herramientas acotado y estable (7 en trabajo, 5 personales).
2. **Perfil Pragmático Estándar (Cluster 1):** Constituye el núcleo mayoritario de la industria (mediana de 11 años de código y 6 de experiencia laboral). Utilizan un conjunto de herramientas equilibrado y orientado al cumplimiento operativo diario (5 en trabajo, 4 personales).
3. **Early Adopters Exploradores (Cluster 2):** Profesionales con experiencia intermedia-alta (mediana de 20 años de código) que destacan por una tasa masiva de experimentación técnica, cuadruplicando la adopción de herramientas tanto en el plano laboral como personal (20 en trabajo, 20 personales).

Tras contrastar los grupos mediante el **Test de Kruskal-Wallis**, se obtuvieron $p\text{-valores} < 0.001$ en todas las variables analizadas, confirmando diferencias estadísticamente significativas en la trayectoria y el patrón de uso de herramientas[cite: 11, 12].

Por consiguiente, **SE RECHAZA LA HIPÓTESIS NULA ($H_0$)** y se valida la **HIPÓTESIS ALTERNATIVA ($H_1$)**, demostrando que la comunidad técnica no se distribuye homogéneamente, sino que está compuesta por arquetipos multidimensionales latentes y claramente diferenciados[cite: 11, 12].

# Hipotesis Planteadas Por La Catedra


## Hipótesis 1: Salario de Desarrolladores en Argentina vs. SMVM

**"Los desarrolladores que respondieron la encuesta de Argentina tienen un salario 3 veces mayor al Salario Mínimo, Vital y Móvil (SMVM)".**

**Población de estudio:** Desarrolladores encuestados residentes en Argentina con registro salarial válido en `ConvertedCompYearly`.

**Hipótesis Nula ($H_0$):** El salario anual medio de los desarrolladores en Argentina es menor o igual a USD 9.720,75 ($\mu \le 9.720,75$).

**Hipótesis Alternativa ($H_1$):** El salario anual medio de los desarrolladores en Argentina es estrictamente mayor a USD 9.720,75 ($\mu > 9.720,75$).

**Atributos involucrados:**
- `ConvertedCompYearly`: Compensación total anual estandarizada en dólares (cuantitativa continua).
- `Country`: País de residencia del encuestado (cualitativa nominal), utilizado como filtro para aislar la muestra de Argentina.

**Justificación y Cálculo del Valor de Referencia:**
Tomando como base el Salario Mínimo, Vital y Móvil (SMVM) de referencia ($398.800 ARS mensuales), se calcula el ingreso anual incorporando el Sueldo Anual Complementario (SAC / 13 meses):
$$\text{SMVM Anualizado} = \$398.800 \times 13 = \$5.184.400 \text{ ARS}$$

Al multiplicar dicho valor por 3 y convertirlo a dólares con un tipo de cambio de referencia de \$1.600 ARS/USD, se obtiene el umbral estandarizado:
$$\text{Umbral de Referencia} = \frac{3 \times \$5.184.400}{1.600} = \text{USD } 9.720,75$$

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df_arg, x='ConvertedCompYearly', bins=30, color='blue')
plt.title('Distribución de Salarios Anuales en Argentina (USD)', fontsize=14)
plt.xlabel('Salario Anual (USD)', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)

plt.show()


### ¿Qué hacemos en este paso?
1. **Inspección de la Distribución:** Construimos un histograma de frecuencias de la variable `ConvertedCompYearly` para la submuestra de desarrolladores en Argentina.
2. **Evaluación de Asimetría:** Analizamos la forma empírica de la distribución para detectar sesgos o presencia de valores atípicos.

### ¿Por qué lo hacemos?
- **Verificación de Supuestos Paramétricos:** Aunque la distribución salarial presenta un claro sesgo positivo (asimetría hacia la derecha), el tamaño muestral obtenido para Argentina supera holgadamente el umbral de $n > 30$ observaciones.
- **Aplicación del Teorema del Límite Central (TLC):** En virtud del TLC, la distribución del promedio muestral ($\bar{X}$) se aproxima a una distribución normal estándar, lo que garantiza la validez y robustez matemática de la prueba $t$ de Student para una muestra.

In [ ]:
salarios_arg = df_arg['ConvertedCompYearly'].dropna()

valor_hipotesis = 9720.75

#  alternative='greater' para evaluar si la media poblacional es MAYOR a 9720.75
t_stat, p_value = stats.ttest_1samp(a=salarios_arg, popmean=valor_hipotesis, alternative='greater')

print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor P (p-value): {p_value:.4e}\n")

if p_value < 0.05:
    print("Conclusión: Rechazamos la hipótesis nula (H0).")
    print("Hay evidencia estadística suficiente para afirmar que el salario promedio de los desarrolladores en Argentina es significativamente mayor a 3 veces el SMVM (USD 9720.75).")
else:
    print("Conclusión: No podemos rechazar la hipótesis nula (H0).")
    print("No hay evidencia estadística para afirmar que el salario promedio es superior a 3 veces el SMVM (USD 9720.75).")

### ¿Qué hacemos en este paso?
1. **Depuración de Faltantes:** Filtramos la columna `ConvertedCompYearly` en la submuestra de Argentina descartando observaciones nulas.
2. **Configuración del Test de Una Cola:** Ejecutamos el test $t$ de Student contrastando la media muestral contra el valor escalar $\mu_0 = 9.720,75$ USD con la condición `alternative='greater'`.
3. **Evaluación de Significancia:** Comparamos el $p\text{-valor}$ resultante frente al nivel de significancia prefijado $\alpha = 0,05$.

### ¿Por qué lo hacemos?
- **Evaluación Unilateral:** La formulación de la Hipótesis Alternativa ($H_1$) busca determinar específicamente si el salario supera de forma significativa el umbral (direccionalidad a la derecha), por lo que corresponde aplicar una prueba de una sola cola.


A partir de los datos analizados sobre la muestra de desarrolladores en Argentina, el test **$t$ de Student para una muestra** arrojó un estadístico $T$ significativamente elevado y un valor de $p < 0,001$ ($p\text{-valor} < 0,05$).

Dado que la probabilidad observada es inferior al nivel de significancia establecido ($\alpha = 0,05$), existe evidencia estadística suficiente para rechazar la hipótesis nula.

Por lo tanto, **SE RECHAZA LA HIPÓTESIS NULA ($H_0$)** y se aprueba la **HIPÓTESIS ALTERNATIVA ($H_1$)**, concluyendo que el salario medio anual de los desarrolladores que responden la encuesta en Argentina es estrictamente mayor a 3 veces el Salario Mínimo, Vital y Móvil anualizado (USD 9.720,75).

## Hipótesis 2: Experiencia Laboral vs. Confianza en la Precisión de la IA

**Enunciado formal:** "Los años de experiencia profesional influyen en la confianza en la precisión de las herramientas de IA".

**Población de estudio:** Desarrolladores encuestados en `df_clean` con respuestas válidas en sus años de experiencia laboral (`WorkExp`) y nivel de confianza en la precisión de la IA (`AIAcc`).

**Hipótesis Nula ($H_0$):** No existe una diferencia estadísticamente significativa en la distribución de los años de experiencia profesional (`WorkExp`) entre los diferentes niveles de confianza en la precisión de las herramientas de IA (`AIAcc`).

**Hipótesis Alternativa ($H_1$):** Existe una diferencia estadísticamente significativa en la distribución de los años de experiencia profesional (`WorkExp`) entre al menos un par de niveles de confianza en la precisión de las herramientas de IA (`AIAcc`).

**Atributos involucrados:**
- `WorkExp`: Años de experiencia laboral profesional (cuantitativa continua).
- `AIAcc`: Nivel de confianza percibido en la precisión de las respuestas generadas por herramientas de IA (cualitativa ordinal con 5 niveles: *Highly distrust*, *Somewhat distrust*, *Neither trust nor distrust*, *Somewhat trust*, *Highly trust*).

**Justificación y Marco Teórico:**
A medida que un profesional acumula años de trayectoria en el sector IT, desarrolla mayor criterio técnico para evaluar el código y detectar alucinaciones o imprecisiones en modelos automatizados. Analizar si la experiencia condiciona la confianza en la IA permite determinar si la adopción escéptica o entusiasta está sesgada por el nivel de seniority del desarrollador.

In [ ]:
categorias_confianza = df_clean['AIAcc'].dropna().unique().tolist()
print(categorias_confianza)

plt.figure(figsize=(10, 6))

# Crear el boxplot
sns.boxplot(
    data=df_clean,
    x='AIAcc',
    y='WorkExp',
    order=categorias_confianza,
    palette='Blues',
    showfliers=False  # Oculta la columna de puntos outlayers
)

plt.title('Distribución de Años de Experiencia según el Nivel de Confianza en la IA', fontsize=14)
plt.xlabel('Nivel de Confianza en las Herramientas de IA (AIAcc)', fontsize=12)
plt.ylabel('Años de Experiencia Profesional (workExp)', fontsize=12)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### ¿Qué hacemos en este paso?
1. **Ordenamiento Ordinal:** Definimos la secuencia lógica del atributo `AIAcc` respetando su escala ordinal nativa (desde desconfianza extrema hasta confianza alta).
2. **Visualización por Categorías (`sns.boxplot`):** Graficamos la distribución de los años de experiencia profesional (`WorkExp`) para cada uno de los 5 niveles de confianza, ocultando los valores atípicos gráficos (`showfliers=False`) para no distorsionar la visualización.

### ¿Por qué lo hacemos?
- **Comparación de Posición Central y Dispersión:** El Boxplot permite comparar visualmente las medianas, los rangos intercuartílicos (IQR) y la asimetría del seniority a lo largo de la escala de confianza.
- **Diagnóstico de Normalidad y Variabilidad:** La variable de años de experiencia suele exhibir un marcado sesgo positivo (asimetría hacia la derecha), lo que invalida el supuesto de normalidad requerida por pruebas paramétricas tradicionales.

In [ ]:
    # Separas los años de experiencia por cada nivel de confianza

    g1 = df_clean[df_clean['AIAcc'] == 'Highly distrust']['WorkExp'].dropna()    
    g2 = df_clean[df_clean['AIAcc'] == 'Somewhat distrust']['WorkExp'].dropna()
    g3 = df_clean[df_clean['AIAcc'] == 'Neither trust nor distrust']['WorkExp'].dropna()
    g4 = df_clean[df_clean['AIAcc'] == 'Somewhat trust']['WorkExp'].dropna()
    g5 = df_clean[df_clean['AIAcc'] == 'Highly trust']['WorkExp'].dropna()

    # Kruskal-Wallis
    estadistico, p_value = stats.kruskal(g1, g2, g3, g4, g5)

    if (p_value) < 0.05:
        print(f"P-value: {p_value}")
        print("Conclusión: Se rechaza H0. Hay diferencias significativas en la experiencia profesional según el nivel de confianza.")
    else:
        print(f"P-value: {p_value}")
        print("Conclusión: No se rechaza H0. No hay diferencias significativas.")

### ¿Qué hacemos en este paso?
1. **Segmentación de Submuestras:** Dividimos la variable cuantitativa `WorkExp` en 5 grupos independientes de observaciones (`g1` a `g5`), correspondientes a cada nivel ordinal de `AIAcc`.
2. **Ejecución del Test de Kruskal-Wallis (`stats.kruskal`):** Evaluamos si las distribuciones subyacentes de experiencia laboral son estocásticamente iguales entre los 5 grupos de confianza.
3. **Evaluación de Significancia:** Contrastamos el $p\text{-valor}$ obtenido contra el nivel de significancia prefijado $\alpha = 0,05$.

### ¿Por qué lo hacemos?
- **Inadecuación del ANOVA Paramétrico:** Como `WorkExp` no se distribuye de manera normal y presenta varianzas heterogéneas, la prueba ANOVA de un factor no resulta metodológicamente adecuada.
- **Robustez No Paramétrica:** El test de Kruskal-Wallis es el equivalente no paramétrico de ANOVA para $k \ge 3$ muestras independientes. Evalúa si las medianas de los grupos provienen de la misma población sin requerir supuestos de normalidad.


A partir de la prueba inferencial no paramétrica de **Kruskal-Wallis** aplicada sobre las 5 submuestras de confianza (`AIAcc`), se obtuvo un estadístico $H$ relevante y un valor de $p < 0,001$ ($p\text{-valor} < 0,05$).

Dado que el $p\text{-valor}$ resulta marcadamente inferior al nivel de significancia elegido ($\alpha = 0,05$), existe evidencia estadística suficiente para rechazar la hipótesis de igualdad entre distribuciones.

Por lo tanto, **SE RECHAZA LA HIPÓTESIS NULA ($H_0$)** y se confirma la **HIPÓTESIS ALTERNATIVA ($H_1$)**, concluyendo que los años de experiencia profesional influyen de manera estadísticamente significativa en el nivel de confianza que los desarrolladores depositan en la precisión de las herramientas de IA.

## Hipotesis 3: 
**Satisfacción laboral y consideración de cambio de rol o carrera**

**Población de estudio:** Desarrolladores encuestados con respuestas válidas en su nivel de satisfacción laboral e intención de cambio de empleo (`df_clean`).

**Hipótesis Nula ($H_0$):** No existe una relación o asociación estadísticamente significativa entre los niveles de satisfacción en el trabajo actual (`JobSat`) y la consideración o intención de un cambio de rol o carrera (`SO_JobSearchStatus`).

**Hipótesis Alternativa ($H_1$):** Existe una relación estadísticamente significativa entre los niveles de satisfacción laboral en el trabajo actual y la consideración o intención de un cambio de rol o carrera.

**Atributos involucrados:**
- `JobSat`: Nivel de satisfacción en el trabajo actual.
- `SO_JobSearchStatus`: Estado de búsqueda de empleo / consideración de cambio de rol.


In [ ]:
# 1. Filtrar registros con respuestas válidas en ambas variables
col_sat = "JobSat"
col_cambio = (
    "SO_JobSearchStatus"
    if "SO_JobSearchStatus" in df_clean.columns
    else "NewRole"
)

df_4c = df_clean.dropna(subset=[col_sat, col_cambio]).copy()

# 2. Generar tabla de contingencia porcentual por nivel de satisfacción
tabla_pct = (
    pd.crosstab(df_4c[col_sat], df_4c[col_cambio], normalize="index") * 100
)

# 3. Gráfico de barras apiladas
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

ax = tabla_pct.plot(
    kind="bar", stacked=True, colormap="tab10", figsize=(10, 6), edgecolor="black"
)

plt.title(
    "Intención de Cambio de Rol según Nivel de Satisfacción Laboral (JobSat)",
    fontsize=13,
    pad=15,
)
plt.xlabel("Nivel de Satisfacción Laboral", fontsize=11)
plt.ylabel("Porcentaje (%)", fontsize=11)
plt.legend(
    title="Búsqueda de Empleo / Cambio de Rol",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=10,
)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()

plt.show()

En el gráfico se observa visualmente que a medida que disminuye el nivel de satisfacción laboral (avanzando hacia categorías de insatisfacción), se incrementa sustancialmente el porcentaje de desarrolladores que buscan activamente un empleo o están abiertos a nuevas oportunidades. Al tratarse de dos variables categóricas (cualitativas), no corresponden las pruebas de comparación de medias (como T-Student o ANOVA). Por lo tanto, aplicaremos el test no paramétrico de **Chi-cuadrado ($\chi^2$) de Independencia** junto con el indicador **V de Cramér** para medir la intensidad del efecto.

In [ ]:
# 1. Matriz de frecuencias absolutas para Chi-cuadrado
tabla_contingencia = pd.crosstab(df_4c[col_sat], df_4c[col_cambio])

# 2. Ejecutar el test de Chi-cuadrado de independencia
chi2_stat, p_value, dof, expected = stats.chi2_contingency(tabla_contingencia)

# 3. Calcular el tamaño del efecto (V de Cramér)
n = tabla_contingencia.sum().sum()
min_dim = min(tabla_contingencia.shape) - 1
cramer_v = np.sqrt(chi2_stat / (n * min_dim))

# 4. Mostrar resultados numéricos
print("--- TEST DE CHI-CUADRADO DE INDEPENDENCIA ---")
print(f"Estadístico Chi2 : {chi2_stat:.4f}")
print(f"Grados de libertad: {dof}")
print(f"Valor p (p-value) : {p_value:.4e}")
print(f"V de Cramér       : {cramer_v:.4f}\n")

# 5. Veredicto final dinámico según p-valor
if p_value < 0.05:
    print("Conclusión: Rechazamos la hipótesis nula (H0).")
    print(
        "Hay evidencia estadística suficiente para afirmar que los niveles de satisfacción laboral "
        "están significativamente relacionados con la consideración de un cambio de rol o carrera."
    )
else:
    print("Conclusión: No podemos rechazar la hipótesis nula (H0).")
    print(
        "No hay evidencia estadística suficiente para afirmar una relación entre la satisfacción laboral "
        "y la consideración de un cambio de rol o carrera."
    )